In [1]:
import rclpy
import math
from rclpy.node import Node

from geometry_msgs.msg import PointStamped
import numpy as np

import random

In [2]:
class PointStampedPublisher(Node):
    def __init__(self):
        super().__init__('point_stamped_publisher')

        self.publisher_ = self.create_publisher(
            PointStamped,
            'point_stamped',
            10
        )

        # 10 Hz = 0.1 seconds
        self.timer = self.create_timer(0.1, self.publish_point)

        # Circle parameters
        self.radius = 0.5        # meters
        self.angular_speed = 1.0 # rad/s
        self.theta = 0.0

        self.origin = [-3.5, 1.35]
        self.noise_std = 0.02  # standard deviation

    def publish_point(self):
        msg = PointStamped()
        msg.header.stamp = self.get_clock().now().to_msg()
        msg.header.frame_id = 'map'

        # Circular motion in XY plane
        msg.point.x = self.radius * math.cos(self.theta)
        msg.point.y = self.radius * math.sin(self.theta)
        msg.point.z = 0.0

        # Add Noise
        noise = np.random.rand(2)
        msg.point.x += random.gauss(self.origin[0], self.noise_std)
        msg.point.y += random.gauss(self.origin[1], self.noise_std)
        
        self.publisher_.publish(msg)

        ## Moving 
        # Advance angle based on timer period (0.1 s)
        self.theta += self.angular_speed * 0.1

        # Keep angle bounded (optional but clean)
        if self.theta > 2 * math.pi:
            self.theta -= 2 * math.pi



In [ ]:
rclpy.init()
node = PointStampedPublisher()
rclpy.spin(node)
node.destroy_node()
rclpy.shutdown()